In [ ]:
from phd.Augmentations.common_code.get_dataset import *
from phd.Augmentations.common_code.models import *
from phd.Augmentations.common_code.utils import *
import tensorflow as tf
import matplotlib.pyplot as plt

training_config = {
"ENABLE_DETERMINISM": False,
"SEED": 17,
"AUTO": tf.data.AUTOTUNE,
"TRAIN_BATCH_SIZE": 256,
"TEST_BATCH_SIZE": 256,
"NUMBER_OF_EPOCHS": 500,
"EARLY_STOPPING_TOLERANCE": 10,
"LEARNING_RATE": 1e-4,
"IMAGE_SIZE": 128,
"SHUFFLE_BUFFER": 128,
"TPU": False,
"AUGMENTATIONS_ROTATE": False,
"AUGMENTATIONS_FLIP_HORIZONTALLY": False,
"AUGMENTATIONS_FLIP_VERTICALLY": False,
"AUGMENTATIONS_ZOOM": False,
"AUGMENTATIONS_SALT_AND_PEPPER": False,
"AUGMENTATIONS_CUTOUT": False,
"AUGMENTATIONS_RANDOM_NOISE": False,
"AUGMENTATIONS_CENTER_NOISE": False,
"AUGMENTATIONS_OUTSIDE_NOISE": False,
"AUGMENTATIONS_PERLIN_CENTER_NOISE": False,
"USE_ADABELIEF_OPTIMIZER": False,
"GALAXY_DATASET": "SDSS",
"TRAIN_DATASET_SIZE": 128,
"TEST_DATASET_SIZE": 128,
"STRETCH": False,
"STEPS_PER_EXECUTION": 1,
"FOLDS": 10,
"NUMBER_OF_CLASSES": 2,
"METRIC": "Accuracy",
"CACHE": False
}

training_config["WANDB_PROJECT_NAME"] = f"Second paper"
training_config["EXPERIMENT_DESCRIPTION"] = f"paper2_4_classes_baseline"
training_config["LOCAL_GCP_PATH_BASE"] = f"./training_models"
training_config["REMOTE_GCP_PATH_BASE"] = f"./training_models"
training_config["DATASET_PATH"] = f"../TFDataset/FourClasses/training"


from photutils.detection import DAOStarFinder
from astropy.stats import sigma_clipped_stats
from astropy.io import fits
import numpy as np
from PIL import Image
from tqdm import tqdm
import warnings
 
from photutils.utils import NoDetectionsWarning

for dataset_name in [
        "Bin_SDSS_z_0.0-0.25",
        "Bin_DES_z_0.0-0.15",
        "Bin_Hubble_z_0.0-0.3",
        "Bin_Hubble_z_0.3-1.0",
        "Bin_Hubble_z_1.0_4.0",
    ]:
    print(f"Dataset: {dataset_name}")
    test_dataset = get_intial_fold_dataset(
        training_config,
        f"TFDataset/{dataset_name}/all/fold_1/train",
        seed = training_config["SEED"],
        shuffle = False,
        cache = False)

    warnings.simplefilter('ignore', category=NoDetectionsWarning)

    count = 0
    stars = []

    for i, (byte_arr, cl) in enumerate((test_dataset)):
        img = np.array(Image.fromarray(np.uint8(byte_arr * 255)).convert("L"))
        mean, median, std = sigma_clipped_stats(img, sigma=3.0)
        daofind = DAOStarFinder(fwhm=3.0, threshold=5.*std)
        sources = daofind.find_stars(img - median)
        number_of_stars = 0 if sources is None else len(sources)
        count += 1
        stars.append(number_of_stars)

        # if i < 100:
        #     plt.imshow(np.uint8(byte_arr * 255))
        #     plt.savefig(f"test_images/{dataset_name}/{i}_{number_of_stars}.jpg")


    stars = np.array(stars)
    print(f"Average: {stars.mean()}, Median: {np.median(stars)}")
    print()